# Temporal Alignment — News and Liquidity Data

Loads all articles from the `articles` table, assigns each to its next available
trading hour in `market_context` using `merge_asof(direction='forward')`, left-joins
`llm_features`, and writes the result to the `liquidity` table.

**Key design decisions:**
- `dt.ceil('h')` preserves causality — never assigns article to hour before publication
- `merge_asof(direction='forward')` vectorized O(n log n) replacement for the old `.apply()` loop
- Inner join to `market_context`: articles outside coverage are dropped and counted
- Left join `llm_features`: articles without LLM scores are kept, flagged with `has_llm_features=0`
- `assignment_gap` records hours between publication timestamp and assigned trading hour
- All reads and writes use `wti_thesis.db` directly — no CSV reads or writes

### General imports

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import datetime as dt

### Load all data from DB

In [2]:
conn = sqlite3.connect("../01_data/wti_thesis.db")

df_articles = pd.read_sql("""
    SELECT id, title, datetime, domain, url, body_valid
    FROM articles
""", conn)

df_mc = pd.read_sql("""
    SELECT datetime_hour, log_volume, price_range, log_return, amihud
    FROM market_context
    WHERE log_volume > 0
""", conn)

df_llm = pd.read_sql("""
    SELECT article_id, sentiment_score, magnitude, event_type,
           entities, certainty, price_direction, time_horizon
    FROM llm_features
""", conn)

conn.close()

df_articles['datetime'] = pd.to_datetime(df_articles['datetime'], utc=True)
df_mc['datetime_hour'] = pd.to_datetime(df_mc['datetime_hour'], utc=True)
df_mc = df_mc.sort_values('datetime_hour').reset_index(drop=True)

print(f"Articles:                        {len(df_articles):,}")
print(f"Market context hours (vol > 0):  {len(df_mc):,}")
print(f"LLM features:                    {len(df_llm):,}")
print(f"Market context range: {df_mc['datetime_hour'].min()} → {df_mc['datetime_hour'].max()}")

Articles:                        22,795
Market context hours (vol > 0):  10,834
LLM features:                    12,024
Market context range: 2024-05-13 12:00:00+00:00 → 2026-05-13 11:00:00+00:00


### Assign each article to its next available trading hour

`dt.ceil('h')` gives the earliest possible honest hour — the article cannot have
influenced traders before it was published. `merge_asof(direction='forward')` then
finds the smallest `datetime_hour >= datetime_ceil`, which forward-assigns
off-hours articles (nights, weekends) to the next market open instead of discarding them.

In [3]:
df_articles = df_articles.sort_values('datetime').reset_index(drop=True)
df_articles['datetime_ceil'] = df_articles['datetime'].dt.ceil('h')

# Forward-assign: smallest datetime_hour >= datetime_ceil (inner join semantics via dropna)
df_aligned = pd.merge_asof(
    df_articles,
    df_mc[['datetime_hour', 'log_volume', 'price_range', 'log_return', 'amihud']],
    left_on='datetime_ceil',
    right_on='datetime_hour',
    direction='forward'
)

total = len(df_aligned)
df_aligned = df_aligned.dropna(subset=['datetime_hour']).reset_index(drop=True)
dropped = total - len(df_aligned)

df_aligned['assignment_gap'] = (
    df_aligned['datetime_hour'] - df_aligned['datetime']
).dt.total_seconds() / 3600

contemp = (df_aligned['assignment_gap'] < 2).sum()
fwd = (df_aligned['assignment_gap'] >= 2).sum()

print(f"Total articles:          {total:,}")
print(f"Aligned (in range):      {len(df_aligned):,}")
print(f"Dropped (out of range):  {dropped:,}")
print(f"Contemporaneous (<2h):   {contemp:,}")
print(f"Forward-assigned (>=2h): {fwd:,}")
print(f"Date range: {df_aligned['datetime_hour'].min()} → {df_aligned['datetime_hour'].max()}")

Total articles:          22,795
Aligned (in range):      22,795
Dropped (out of range):  0
Contemporaneous (<2h):   15,290
Forward-assigned (>=2h): 7,505
Date range: 2024-05-13 12:00:00+00:00 → 2026-05-12 01:00:00+00:00


### Sanity checks

In [4]:
BASELINE_ALIGNED = 13_690

assert len(df_aligned) >= BASELINE_ALIGNED, (
    f"STOP: {len(df_aligned):,} aligned rows — expected >= {BASELINE_ALIGNED:,}"
)
assert len(df_aligned) <= total, (
    f"STOP: {len(df_aligned):,} aligned > {total:,} total — merge error"
)
assert df_aligned['log_volume'].isna().sum() == 0, (
    f"STOP: {df_aligned['log_volume'].isna().sum()} rows with NULL log_volume"
)
print(f"Sanity checks passed — {len(df_aligned):,} aligned rows")

Sanity checks passed — 22,795 aligned rows


### Left-join LLM features

In [5]:
df_aligned = df_aligned.merge(
    df_llm,
    left_on='id',
    right_on='article_id',
    how='left'
)
df_aligned['has_llm_features'] = df_aligned['sentiment_score'].notna().astype(int)

with_llm = df_aligned['has_llm_features'].sum()
without_llm = len(df_aligned) - with_llm
print(f"With LLM features:    {with_llm:,}")
print(f"Without LLM features: {without_llm:,}")

With LLM features:    12,024
Without LLM features: 10,771


### Write liquidity table to DB

In [6]:
df_liquidity = df_aligned[[
    'id', 'datetime_hour', 'assignment_gap',
    'log_volume', 'price_range', 'log_return', 'amihud',
    'sentiment_score', 'magnitude', 'event_type', 'entities',
    'certainty', 'price_direction', 'time_horizon', 'has_llm_features'
]].copy().rename(columns={'id': 'article_id'})

df_liquidity['datetime_hour'] = df_liquidity['datetime_hour'].astype(str)

conn = sqlite3.connect("../01_data/wti_thesis.db")
conn.execute("DROP TABLE IF EXISTS liquidity")
conn.commit()
df_liquidity.to_sql('liquidity', conn, if_exists='replace', index=False)
count = conn.execute("SELECT COUNT(*) FROM liquidity").fetchone()[0]
conn.close()

print(f"Liquidity table written: {count:,} rows")

Liquidity table written: 22,795 rows


### Verify

In [7]:
conn = sqlite3.connect("../01_data/wti_thesis.db")

stats = pd.read_sql("""
    SELECT
        COUNT(*)                                              AS total_aligned,
        SUM(CASE WHEN assignment_gap < 2  THEN 1 ELSE 0 END) AS contemporaneous,
        SUM(CASE WHEN assignment_gap >= 2 THEN 1 ELSE 0 END) AS forward_assigned,
        SUM(has_llm_features)                                 AS with_llm,
        SUM(1 - has_llm_features)                             AS without_llm,
        MIN(datetime_hour)                                    AS date_min,
        MAX(datetime_hour)                                    AS date_max
    FROM liquidity
""", conn)
print(stats.to_string())

sample = pd.read_sql("""
    SELECT a.title, a.datetime, l.datetime_hour, l.log_volume, l.assignment_gap
    FROM articles a
    JOIN liquidity l ON a.id = l.article_id
    ORDER BY a.datetime DESC
    LIMIT 5
""", conn)
print(sample.to_string())

conn.close()
print("\nAlignment complete.")

   total_aligned  contemporaneous  forward_assigned  with_llm  without_llm                   date_min                   date_max
0          22795            15290              7505     12024        10771  2024-05-13 12:00:00+00:00  2026-05-12 01:00:00+00:00
                                                                                                            title                   datetime              datetime_hour  log_volume  assignment_gap
0                                             Letters : Supporting social workers will help keep foster kids safe  2026-05-12 00:30:00+00:00  2026-05-12 01:00:00+00:00    7.760041            0.50
1                                                 Saudi Aramco CEO warns oil markets may not normalize until 2027  2026-05-12 00:30:00+00:00  2026-05-12 01:00:00+00:00    7.760041            0.50
2                                                                              Remittances expose Gulf dependency  2026-05-11 23:30:00+00:00  2026-05-12 0

### Update project logbook

In [ ]:
entry = f"""
---

## Phase 5 — Notebook 05 DB Migration ({dt.date.today()})

### Alignment rewrite — CSV → SQLite

**Changes:**
- All reads now come from `wti_thesis.db` (articles, market_context, llm_features) — no CSV reads
- Replaced slow `apply(get_next_trading_hour)` loop with vectorized `merge_asof(direction='forward')`
- Inner join articles → market_context: articles beyond coverage are dropped and counted
- Left join llm_features → `has_llm_features` flag added
- Sanity check: assert aligned >= 13,690 before writing
- `liquidity` table now includes LLM feature columns (denormalized for TFT training)
- `DROP TABLE IF EXISTS liquidity` before write — no stale rows

**Results after rewrite:**
- Total articles in DB: {total:,}
- Aligned to market_context: {len(df_aligned):,}
- Dropped (out of range): {dropped:,}
- Contemporaneous (<2h gap): {contemp:,}
- Forward-assigned (>=2h): {fwd:,}
- With LLM features: {with_llm:,}
- Without LLM features: {without_llm:,}
"""

logbook_path = "../05_reports/thesis/project_logbook.md"
with open(logbook_path, 'a') as f:
    f.write(entry)

print(f"Logbook updated: {logbook_path}")